<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI_Phase19_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from datetime import datetime,timezone
from collections import Counter
from decimal import Decimal,InvalidOperation
import os,json,hashlib,sqlite3,copy,subprocess,tempfile,ast,re

try:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
except ImportError:
    pass
MYDRIVE=Path(os.environ.get('ICHI2027_MYDRIVE','/content/drive/MyDrive')).expanduser().resolve()
assert MYDRIVE.is_dir(), 'Mount the same Google Drive used in Phase18 or define ICHI2027_MYDRIVE.'
PROJECT=MYDRIVE/'NeuroFHIR_ICHI2027'
OUT=PROJECT/'phase19_outputs'
PRIVATE=OUT/'_PRIVATE_DO_NOT_SHARE'
PRIVATE.mkdir(parents=True,exist_ok=True)
PUBLIC_PATH=OUT/'ICHI2027_Phase19_SHAREABLE.json'
PRIVATE_PATH=PRIVATE/'ICHI2027_Phase19_PRIVATE_D_AUDIT_MANIFEST.json'
PRIVATE_EVENTS=PRIVATE/'ICHI2027_Phase19_REAL_D_CASE_AUDIT_PRIVATE.ndjson'
FEATURES=('LHIPPOC','RHIPPOC','LENTORHIN','RENTORHIN','LMIDTEMP','RMIDTEMP','VENTRICLES','ICV')
BIO_FEATURES=tuple(x for x in FEATURES if x!='ICV')
STATES={'TRUSTED','CAUTION','REVIEW_REQUIRED'}
FIELD_SYSTEM='https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-field'
PAIR_PREFIX='Research-only TRACE Q90 longitudinal subject-pair reliability for '
PACKAGE_TEXT='Research-only descriptive TRACE subject-pair package classification'

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for part in iter(lambda:f.read(1024*1024),b''):h.update(part)
    return h.hexdigest()
def timestamp():return datetime.now(timezone.utc).isoformat()
def dec(v):
    x=Decimal(str(v))
    assert x.is_finite()
    return x

def canonical_dec(v):return format(dec(v).normalize(),'f')

def flagset(flags):return sorted({x['rule'] for x in flags})

print('Phase19: REAL candidate A/B imaging-specific D audit + separately labeled SYNTHETIC E0-E7 fixtures; no browser uploads.')


Mounted at /content/drive
Phase19: REAL candidate A/B imaging-specific D audit + separately labeled SYNTHETIC E0-E7 fixtures; no browser uploads.


In [2]:
public18_path=Path(os.environ.get('ICHI2027_PHASE18_JSON',str(PROJECT/'phase18_outputs'/'ICHI2027_Phase18_SHAREABLE.json')))
private18_path=Path(os.environ.get('ICHI2027_PHASE18_PRIVATE_MANIFEST',str(PROJECT/'phase18_outputs'/'_PRIVATE_DO_NOT_SHARE'/'ICHI2027_Phase18_PRIVATE_AB_MANIFEST.json')))
public17_path=PROJECT/'phase17_outputs'/'ICHI2027_Phase17_SHAREABLE.json'
private17_path=PROJECT/'phase17_outputs'/'_PRIVATE_DO_NOT_SHARE'/'ICHI2027_Phase17_PRIVATE_LINKAGE_MANIFEST.json'
for p in (public18_path,private18_path,public17_path,private17_path):
    assert p.is_file(),f'Restore original PRIVATE Drive artifact or shareable JSON: {p.name}. Do not upload private files here.'
p18=json.loads(public18_path.read_text(encoding='utf-8'))
m18=json.loads(private18_path.read_text(encoding='utf-8'))
p17=json.loads(public17_path.read_text(encoding='utf-8'))
m17=json.loads(private17_path.read_text(encoding='utf-8'))
assert p18['phase']=='18_real_private_two_arm_A_B_ablation' and m18['phase']==p18['phase']
assert m18['run_utc']==p18['run_utc']
assert sha256_file(public17_path)==p18['phase17_public_json_sha256']==m18['phase17_public_json_SHA256']
assert m18['arm_contract_sha256']==p18['arm_contract_sha256']
assert p18['direct_A_vs_B_invariants']['numeric_event_key_and_exact_Decimal_value_mismatches']==0
assert p18['direct_A_vs_B_invariants']['structural_cohort_membership_symmetric_difference']==0
assert p17['fhir_r4']['official_schema_validation']=='PASS'
assert m17['private_FHIR_NDJSON_sha256']==m18['source_private_FHIR_SHA256']
assert m17['original_source_notebook_sha256']==m18['original_TRACE_20b_notebook_sha256']
assert m17['original_replay_calibration_csv_sha256_PRIVATE']==m18['regenerated_calibration_SHA256']
assert m17['run_utc']==m18['original_phase17_run_utc']
FHIR_PATH=Path(os.environ.get('ICHI2027_PHASE17_PRIVATE_FHIR',m17['private_FHIR_NDJSON']))
DB=Path(os.environ.get('ICHI2027_PHASE18_PRIVATE_DB',m18['private_A_B_SQLite_file']))
assert FHIR_PATH.is_file() and sha256_file(FHIR_PATH)==m18['source_private_FHIR_SHA256']
assert DB.is_file() and sha256_file(DB)==m18['private_A_B_SQLite_SHA256']
print('Exact Phase17 real-FHIR / Phase18 A/B private-file attestation PASS; original source and calibration lineage unchanged.')


Exact Phase17 real-FHIR / Phase18 A/B private-file attestation PASS; original source and calibration lineage unchanged.


In [3]:
# Freeze these rules before inspecting real A/B case outcomes. Source-unknown dimensions are
# INDETERMINATE, never counted as transformation-induced loss or silently imputed.
D_CONTRACT={
 'method':'SOURCE_TARGET_NEUROIMAGING_EVIDENCE_GRAPH_AUDIT_PROTOTYPE_V1',
 'version':'2026-09-28-D1',
 'comparison_unit':'person_bundle; repeated measurements stay nested, not independent cases',
 'source_ground_truth':'actual_phase17_FHIR_R4+attested_PHASE16_replay_or_fully_synthetic_fixture',
 'event_identity':'source_FHIR_fullUrl_or_synthetic_fixture_id',
 'normalization':{'numeric':'exact_decimal','unit_conversions':{'mm3_to_cm3':'divide_by_1000','cm3_to_mm3':'multiply_by_1000'},
                  'units_missing_at_source':'NOT_EVALUABLE_NOT_AN_ETL_FAILURE'},
 'required_preservation_dimensions':['event_cardinality','numeric_value','source_verified_unit_if_present',
   'laterality_and_anatomy','person_and_date','image_reference','unverified_source_identifier_as_source_label',
   'BL_M12_pair_links','original_TRACE_pair_state','regenerated_calibration_reference',
   'TRACE_package_state_and_member_links','software_identity_only_if_source_attests'],
 'rule_codes':[
  'MISSING_NUMERIC_EVENT','EXTRA_NUMERIC_EVENT','DUPLICATE_NUMERIC_EVENT_ID',
  'NUMERIC_VALUE_CHANGED','UNIT_LOSS_OR_UNRESOLVED_CONVERSION','UNSUPPORTED_TARGET_UNIT_CLAIM',
  'ANATOMY_LATERALITY_MISMATCH','PERSON_OR_DATE_OR_TIMEPOINT_CHANGED',
  'IMAGE_EVENT_LOSS','IMAGE_SOURCE_LABEL_LOSS','IMAGE_LINK_LOSS_OR_MISLINK','BROKEN_TARGET_IMAGE_REFERENCE',
  'DUPLICATE_IMAGE_EVENT_ID','SOURCE_SOFTWARE_VERSION_LOSS',
  'TRACE_PAIR_EVENT_LOSS','TRACE_PAIR_LINK_OR_TIMEPOINT_ERROR','TRACE_STATE_LOSS_OR_CHANGE',
  'TRACE_CALIBRATION_REFERENCE_LOSS_OR_CHANGE','TRACE_PACKAGE_EVENT_LOSS',
  'TRACE_PACKAGE_STATE_OR_MEMBERSHIP_CHANGED','DUPLICATE_TRACE_PAIR_EVENT_ID'],
 'limitations':['source comparison detects contract violations not accuracy of the source data',
   'the deliberately ablated B arm is not the standard OMOP comparator',
   'MRI-specific D is a research prototype not a validated clinical safety system',
   'actual real MRI source units and DICOM UID semantics remain unverified',
   'original TRACE states are regenerated pair-level not historical source-frozen scan-level',
   'synthetic fixtures and direct A/B ablation are not natural fault prevalence or independent diagnostic accuracy']}
D_SHA256=hashlib.sha256(json.dumps(D_CONTRACT,sort_keys=True,separators=(',',':')).encode()).hexdigest()
RULES=set(D_CONTRACT['rule_codes'])
print('Imaging-specific D method contract frozen:',D_SHA256)


Imaging-specific D method contract frozen: 76082c54d3795d423c2d9632282e21b9eb894dd4b33668ce63ea2744001ac58e


In [4]:
def event_index(rows,kind,issues):
    idx={}
    for item in rows:
        key=item['id']
        if key in idx:
            rule={'volumes':'DUPLICATE_NUMERIC_EVENT_ID','images':'DUPLICATE_IMAGE_EVENT_ID','pairs':'DUPLICATE_TRACE_PAIR_EVENT_ID'}[kind]
            issues.append({'rule':rule,'evidence':kind})
        else:idx[key]=item
    return idx

def unit_normalize(target_value,target_unit,source_unit):
    x=dec(target_value)
    if source_unit==target_unit:return x,False
    if source_unit=='mm3' and target_unit=='cm3':return x*Decimal(1000),True
    if source_unit=='cm3' and target_unit=='mm3':return x/Decimal(1000),True
    return None,False

def audit_graph(source,target):
    issues=[]; unknown=Counter(); normalized=0
    def report(code,level,identity=None):
        assert code in RULES
        # identity is only ever written into a PRIVATE Drive file; public report contains counts only
        issues.append({'rule':code,'level':level,'identity':identity})
    si=event_index(source['images'],'images',issues);ti=event_index(target['images'],'images',issues)
    sv=event_index(source['volumes'],'volumes',issues);tv=event_index(target['volumes'],'volumes',issues)
    sp=event_index(source['pairs'],'pairs',issues);tp=event_index(target['pairs'],'pairs',issues)
    for k,src in si.items():
        tgt=ti.get(k)
        if tgt is None:
            report('IMAGE_EVENT_LOSS','image',k);continue
        if src.get('source_identifiers') and src['source_identifiers']!=tgt.get('source_identifiers'):
            report('IMAGE_SOURCE_LABEL_LOSS','image',k)
        src_sw=src.get('software_version')
        if src_sw is None:unknown['source_software_version_unverified']+=1
        elif src_sw!=tgt.get('software_version'):report('SOURCE_SOFTWARE_VERSION_LOSS','image',k)
    for k,src in sv.items():
        tgt=tv.get(k)
        if tgt is None:
            report('MISSING_NUMERIC_EVENT','numeric',k);continue
        for x in ('person','feature','timepoint','date'):
            if x=='feature':
                if src[x]!=tgt.get(x):report('ANATOMY_LATERALITY_MISMATCH','numeric',k)
            elif src[x]!=tgt.get(x):report('PERSON_OR_DATE_OR_TIMEPOINT_CHANGED','numeric',k)
        src_u=src.get('unit');tgt_u=tgt.get('unit')
        if src_u is None:
            unknown['source_numeric_unit_unverified']+=1
            if tgt_u is not None:report('UNSUPPORTED_TARGET_UNIT_CLAIM','numeric',k)
            if dec(src['value'])!=dec(tgt['value']):report('NUMERIC_VALUE_CHANGED','numeric',k)
        elif tgt_u is None:report('UNIT_LOSS_OR_UNRESOLVED_CONVERSION','numeric',k)
        else:
            value,was_normalized=unit_normalize(tgt['value'],tgt_u,src_u)
            if value is None:report('UNIT_LOSS_OR_UNRESOLVED_CONVERSION','numeric',k)
            elif value!=dec(src['value']):report('NUMERIC_VALUE_CHANGED','numeric',k)
            elif was_normalized:normalized+=1
        if src.get('image_id'):
            if tgt.get('image_id')!=src['image_id']:
                report('IMAGE_LINK_LOSS_OR_MISLINK','numeric',k)
            elif tgt['image_id'] not in ti:
                report('BROKEN_TARGET_IMAGE_REFERENCE','numeric',k)
        if tgt.get('image_id') and tgt['image_id'] not in ti:
            if tgt.get('image_id')!=src.get('image_id'):
                report('BROKEN_TARGET_IMAGE_REFERENCE','numeric',k)
    for k in tv.keys()-sv.keys():report('EXTRA_NUMERIC_EVENT','numeric',k)
    for k,src in sp.items():
        tgt=tp.get(k)
        if tgt is None:report('TRACE_PAIR_EVENT_LOSS','pair',k);continue
        for x in ('person','feature','bl','m12'):
            if tgt.get(x)!=src.get(x):
                report('TRACE_PAIR_LINK_OR_TIMEPOINT_ERROR','pair',k);break
        if tgt.get('bl') not in tv or tgt.get('m12') not in tv:
            report('TRACE_PAIR_LINK_OR_TIMEPOINT_ERROR','pair',k)
        if tgt.get('state')!=src.get('state'):report('TRACE_STATE_LOSS_OR_CHANGE','pair',k)
        if tgt.get('calibration')!=src.get('calibration'):
            report('TRACE_CALIBRATION_REFERENCE_LOSS_OR_CHANGE','pair',k)
    sk=source.get('package');tk=target.get('package')
    if sk is not None:
        if tk is None:report('TRACE_PACKAGE_EVENT_LOSS','package',sk['id'])
        elif tk.get('state')!=sk['state'] or set(tk.get('members',[]))!=set(sk['members']):
            report('TRACE_PACKAGE_STATE_OR_MEMBERSHIP_CHANGED','package',sk['id'])
    if source.get('verified_dicom_uids') is not True:
        unknown['source_DICOM_UID_semantics_unverified']+=1
    return {'issues':issues,'issue_counts':Counter(x['rule'] for x in issues),
            'unknown':unknown,'known_unit_conversions_normalized':normalized}


In [5]:
# All synthetic case ground truth is declared here, before tests and before real data audit.
# SYNTHETIC IDs/measurements below are artificial software fixtures, never ADNI data.
def make_synthetic():
    images=[{'id':f'SYN-IMG-{tp}','person':'SYN-PERSON-1','date':date,
             'source_identifiers':[{'system':'SYNTHETIC-ONLY','value':f'SYN-{tp}'}],
             'software_version':'SYNTHETIC-5.1'}
            for tp,date in (('BL','2022-01-01'),('M12','2023-01-01'))]
    volumes=[];pairs=[]
    for tp,date in (('BL','2022-01-01'),('M12','2023-01-01')):
        for idx,feature in enumerate(FEATURES):
            volumes.append({'id':f'SYN-VOL-{tp}-{feature}','person':'SYN-PERSON-1',
                'date':date,'timepoint':tp,'feature':feature,
                'value':str(1000+(idx*100)+(10 if tp=='M12' else 0)),
                'unit':'mm3','image_id':f'SYN-IMG-{tp}'})
    for feature in BIO_FEATURES:
        pairs.append({'id':f'SYN-PAIR-{feature}','person':'SYN-PERSON-1','feature':feature,
         'bl':f'SYN-VOL-BL-{feature}','m12':f'SYN-VOL-M12-{feature}',
         'state':'TRUSTED','calibration':'SYNTHETIC-FROZEN-CALIBRATION'})
    return {'images':images,'volumes':volumes,'pairs':pairs,
            'package':{'id':'SYN-PACKAGE','state':'TRUSTED','members':[p['id'] for p in pairs]},
            'verified_dicom_uids':True}

SYN_SOURCE=make_synthetic()
SYN_TARGETS={}
SYN_EXPECTED={
 'E0_clean':set(),
 'E1_laterality_swap':{'ANATOMY_LATERALITY_MISMATCH'},
 'E2a_known_unit_conversion':set(),
 'E2b_missing_unit':{'UNIT_LOSS_OR_UNRESOLVED_CONVERSION'},
 'E3_wrong_longitudinal_pair':{'TRACE_PAIR_LINK_OR_TIMEPOINT_ERROR'},
 'E4_missing_software_version':{'SOURCE_SOFTWARE_VERSION_LOSS'},
 'E5_stripped_TRACE_state':{'TRACE_STATE_LOSS_OR_CHANGE'},
 'E6_duplicate_and_broken_source_link':{'DUPLICATE_NUMERIC_EVENT_ID','BROKEN_TARGET_IMAGE_REFERENCE','IMAGE_LINK_LOSS_OR_MISLINK'},
 'E7_heldout_combination':{'ANATOMY_LATERALITY_MISMATCH','SOURCE_SOFTWARE_VERSION_LOSS','UNIT_LOSS_OR_UNRESOLVED_CONVERSION'}}
SYN_TARGETS['E0_clean']=copy.deepcopy(SYN_SOURCE)
SYN_TARGETS['E1_laterality_swap']=copy.deepcopy(SYN_SOURCE)
next(x for x in SYN_TARGETS['E1_laterality_swap']['volumes'] if x['id']=='SYN-VOL-BL-LENTORHIN')['feature']='RENTORHIN'
SYN_TARGETS['E2a_known_unit_conversion']=copy.deepcopy(SYN_SOURCE)
next(x for x in SYN_TARGETS['E2a_known_unit_conversion']['volumes'] if x['id']=='SYN-VOL-BL-LHIPPOC').update(value='1',unit='cm3')
SYN_TARGETS['E2b_missing_unit']=copy.deepcopy(SYN_SOURCE)
next(x for x in SYN_TARGETS['E2b_missing_unit']['volumes'] if x['id']=='SYN-VOL-BL-LHIPPOC')['unit']=None
SYN_TARGETS['E3_wrong_longitudinal_pair']=copy.deepcopy(SYN_SOURCE)
p=next(x for x in SYN_TARGETS['E3_wrong_longitudinal_pair']['pairs'] if x['feature']=='LHIPPOC')
p['bl'],p['m12']=p['m12'],p['bl']
SYN_TARGETS['E4_missing_software_version']=copy.deepcopy(SYN_SOURCE)
SYN_TARGETS['E4_missing_software_version']['images'][0]['software_version']=None
SYN_TARGETS['E5_stripped_TRACE_state']=copy.deepcopy(SYN_SOURCE)
SYN_TARGETS['E5_stripped_TRACE_state']['pairs'][0]['state']=None
SYN_TARGETS['E6_duplicate_and_broken_source_link']=copy.deepcopy(SYN_SOURCE)
SYN_TARGETS['E6_duplicate_and_broken_source_link']['volumes'].append(copy.deepcopy(SYN_TARGETS['E6_duplicate_and_broken_source_link']['volumes'][0]))
SYN_TARGETS['E6_duplicate_and_broken_source_link']['volumes'][0]['image_id']='SYN-NOT-AN-IMAGE'
SYN_TARGETS['E7_heldout_combination']=copy.deepcopy(SYN_SOURCE)
SYN_TARGETS['E7_heldout_combination']['volumes'][0].update(feature='RHIPPOC',unit=None)
SYN_TARGETS['E7_heldout_combination']['images'][1]['software_version']=None
SYN_FREEZE_HASH=hashlib.sha256(json.dumps({'source':SYN_SOURCE,'targets':SYN_TARGETS,
  'expected':{k:sorted(v) for k,v in SYN_EXPECTED.items()}},sort_keys=True).encode()).hexdigest()
SYN_RESULTS={}
for name,fixture in SYN_TARGETS.items():
    actual=audit_graph(SYN_SOURCE,fixture)
    got=set(actual['issue_counts'])
    assert got==SYN_EXPECTED[name],f'{name}: expected {SYN_EXPECTED[name]} but got {got}'
    if name=='E2a_known_unit_conversion':assert actual['known_unit_conversions_normalized']==1
    SYN_RESULTS[name]={'expected_rules':sorted(SYN_EXPECTED[name]),'detected_rules':sorted(got),
        'test_pass':True,'normalized_known_unit_conversions':actual['known_unit_conversions_normalized']}
assert len(SYN_RESULTS)==9
print('Synthetic-only E0-E7 incl. conversion/missing-unit split: 9/9 prespecified unit tests PASS. This is NOT real clinical validation.')


Synthetic-only E0-E7 incl. conversion/missing-unit split: 9/9 prespecified unit tests PASS. This is NOT real clinical validation.


In [6]:
# Normalize each actual source FHIR bundle and its actual Phase18 SQLite A/B target rows
# into the SAME frozen D contract used for the artificial fixtures.
def fhir_case(bundle,method_sha,calibration_sha):
    entries={e['fullUrl']:e['resource'] for e in bundle['entry']}
    assert len(entries)==27
    people=[(k,v) for k,v in entries.items() if v['resourceType']=='Patient']
    assert len(people)==1
    person=people[0][0]
    images=[];volumes=[];pairs=[];package=[]
    for k,v in entries.items():
        kind=v['resourceType']
        if kind=='ImagingStudy':
            assert v['subject']['reference']==person
            labels=v.get('identifier',[])
            assert len(labels)==2
            images.append({'id':k,'person':person,'date':v['started'],
                           'source_identifiers':labels,'software_version':None})
        if kind!='Observation':continue
        assert v['subject']['reference']==person
        if 'valueQuantity' in v:
            field=[i['value'] for i in v.get('identifier',[]) if i.get('system')==FIELD_SYSTEM]
            assert len(field)==1
            tp,feature=field[0].split('_',1)
            refs=[x['reference'] for x in v.get('derivedFrom',[])]
            assert tp in ('BL','M12') and feature in FEATURES and len(refs)==1
            assert 'unit' not in v['valueQuantity'] and 'code' not in v['valueQuantity']
            volumes.append({'id':k,'person':person,'timepoint':tp,'feature':feature,
                'date':v['effectiveDateTime'],'value':canonical_dec(v['valueQuantity']['value']),
                'unit':None,'image_id':refs[0]})
        elif v.get('code',{}).get('text','').startswith(PAIR_PREFIX):
            feature=v['code']['text'][len(PAIR_PREFIX):]
            assert feature in BIO_FEATURES
            refs=[x['reference'] for x in v['derivedFrom']]
            assert len(refs)==2
            pairs.append({'id':k,'person':person,'feature':feature,'bl':refs[0],
                'm12':refs[1],'state':v['valueCodeableConcept']['text'],
                'calibration':calibration_sha})
        elif v.get('code',{}).get('text','')==PACKAGE_TEXT:
            package.append({'id':k,'state':v['valueCodeableConcept']['text'],
                            'members':[x['reference'] for x in v['derivedFrom']]})
        else:raise AssertionError('Unexpected FHIR source observation category')
    assert (len(images),len(volumes),len(pairs),len(package))==(2,16,7,1)
    assert {x['state'] for x in pairs}<=STATES
    return {'images':images,'volumes':volumes,'pairs':pairs,'package':package[0],
            'verified_dicom_uids':False}

def sql_targets(conn,person):
    A={'images':[],'volumes':[],'pairs':[],'package':None,'verified_dicom_uids':False}
    B={'images':[],'volumes':[],'pairs':[],'package':None,'verified_dicom_uids':False}
    assert conn.execute('SELECT COUNT(*) FROM A_PERSON WHERE person_key=?',(person,)).fetchone()[0]==1
    assert conn.execute('SELECT COUNT(*) FROM B_PERSON WHERE person_key=?',(person,)).fetchone()[0]==1
    for row in conn.execute('SELECT image_key,person_key,image_date,source_identifiers_JSON FROM A_IMAGE WHERE person_key=?',(person,)):
        key,p,day,labels=row
        A['images'].append({'id':key,'person':p,'date':day,'source_identifiers':json.loads(labels),
                            'software_version':None})
    for row in conn.execute('SELECT observation_key,person_key,image_key,timepoint,feature,observation_date,value_decimal FROM A_NUMERIC WHERE person_key=?',(person,)):
        k,p,img,tp,f,day,value=row
        A['volumes'].append({'id':k,'person':p,'image_id':img,'timepoint':tp,'feature':f,
                             'date':day,'value':canonical_dec(value),'unit':None})
    for row in conn.execute('SELECT observation_key,person_key,timepoint,feature,observation_date,value_decimal FROM B_NUMERIC WHERE person_key=?',(person,)):
        k,p,tp,f,day,value=row
        B['volumes'].append({'id':k,'person':p,'image_id':None,'timepoint':tp,'feature':f,
                             'date':day,'value':canonical_dec(value),'unit':None})
    for row in conn.execute('SELECT pair_key,person_key,feature,baseline_observation_key,month12_observation_key,state,regenerated_calibration_CSV_SHA256 FROM A_PAIR_STATE WHERE person_key=?',(person,)):
        k,p,f,bl,m12,state,calibration=row
        A['pairs'].append({'id':k,'person':p,'feature':f,'bl':bl,'m12':m12,'state':state,
                           'calibration':calibration})
    pkg=conn.execute('SELECT package_key,state FROM A_PACKAGE_STATE WHERE person_key=?',(person,)).fetchone()
    if pkg:
        A['package']={'id':pkg[0],'state':pkg[1],
            'members':[r[0] for r in conn.execute('SELECT pair_key FROM A_PACKAGE_MEMBER WHERE package_key=?',(pkg[0],))]}
    assert (len(A['images']),len(A['volumes']),len(A['pairs']),len(B['volumes']))==(2,16,7,16)
    assert A['package'] is not None
    return A,B


In [7]:
# Apply the frozen D rulebook to REAL transformations; no threshold or rule is re-fitted.
REAL_COUNTERS={'A':Counter(),'B':Counter()};UNKNOWN={'A':Counter(),'B':Counter()}
REAL_FOUNDED={'A':Counter(),'B':Counter()}
actual_people=0
partial_private_path=PRIVATE_EVENTS.with_suffix('.ndjson.part')
if partial_private_path.exists():partial_private_path.unlink()
con=sqlite3.connect(f'file:{DB}?mode=ro',uri=True)
assert con.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
assert con.execute('PRAGMA foreign_key_check').fetchall()==[]
try:
    with FHIR_PATH.open('r',encoding='utf-8') as f,partial_private_path.open('w',encoding='utf-8') as private_output:
        for line in f:
            if not line.strip():continue
            bundle=json.loads(line)
            src=fhir_case(bundle,m17['original_source_notebook_sha256'],m17['original_replay_calibration_csv_sha256_PRIVATE'])
            person=src['volumes'][0]['person']
            A,B=sql_targets(con,person)
            for label,tgt in (('A',A),('B',B)):
                result=audit_graph(src,tgt)
                REAL_COUNTERS[label].update(result['issue_counts'])
                UNKNOWN[label].update(result['unknown'])
                for code in set(result['issue_counts']):REAL_FOUNDED[label][code]+=1
                assert result['known_unit_conversions_normalized']==0
                # Raw identifying row-level material never leaves private Drive.
                private_output.write(json.dumps({'person_key':person,'arm':label,
                    'issues':result['issues'],'source_unknowns':dict(result['unknown'])},
                    sort_keys=True,allow_nan=False)+'\n')
            actual_people+=1
finally:con.close()
assert actual_people==p18['real_input_counts']['patients']==258
# Predeclared A/B transformations must not change real numeric content.
assert sum(REAL_COUNTERS['A'].values())==0,f'Unexpected real A fidelity violations: {REAL_COUNTERS["A"]}'
assert REAL_COUNTERS['B']==Counter({'IMAGE_EVENT_LOSS':516,'IMAGE_LINK_LOSS_OR_MISLINK':4128,
  'TRACE_PAIR_EVENT_LOSS':1806,'TRACE_PACKAGE_EVENT_LOSS':258}),REAL_COUNTERS['B']
assert UNKNOWN['A']['source_numeric_unit_unverified']==4128 and UNKNOWN['B']['source_numeric_unit_unverified']==4128
assert UNKNOWN['A']['source_DICOM_UID_semantics_unverified']==258
assert UNKNOWN['B']['source_DICOM_UID_semantics_unverified']==258
os.replace(partial_private_path,PRIVATE_EVENTS)
print('REAL D contract applied to attested A/B: no unexpected A loss; B specified losses detected; no original-data unknowns turned into invented failures.')


REAL D contract applied to attested A/B: no unexpected A loss; B specified losses detected; no original-data unknowns turned into invented failures.


In [8]:
# Freeze public aggregate outputs separately from participant-level records and private file hashes.
REAL_SAFETY={
 'source_unit_verification':'NOT_AVAILABLE_IN_CURRENT_REAL_SOURCE',
 'source_DICOM_UID_attestation':'NOT_AVAILABLE_IN_CURRENT_REAL_SOURCE',
 'original_ADNI_raw_export':'NOT_AVAILABLE_FROM_THE_PRIOR_VALIDATED_SOURCE_PACKAGE',
 'baseline_MCI_true_label':'NOT_VALIDATED_FOR_CURRENT_REAL_SOURCE',
 'official_micdm':'NOT_RUN',
 'generic_frozen_tfl_v15_comparator_C':'NOT_EXECUTED_ON_CURRENT_MRI',
 'D_prototype':'APPLIED_TO_REAL_PRIVATE_CANDIDATES_A_AND_B_WITH_ORACLE_SOURCE',
 'clinical_qualification_and_false_exclusion':'NOT_EVALUABLE_WITH_UNVERIFIED_SOURCE_UNIT_AND_STANDARD_MAPPING'}
PRIVATE_MANIFEST={
 'project':p18['project'],'phase':'19_frozen_neuro_D_real_AB_and_synthetic_E0_E7',
 'run_utc':timestamp(),'source_phase18_public_SHA256':sha256_file(public18_path),
 'private_phase18_manifest_SHA256':sha256_file(private18_path),
 'source_phase17_FHIR_SHA256':sha256_file(FHIR_PATH),
 'phase18_AB_SQLite_SHA256':sha256_file(DB),
 'neuro_D_contract':D_CONTRACT,'neuro_D_contract_SHA256':D_SHA256,
 'synthetic_fixture_freeze_SHA256':SYN_FREEZE_HASH,
 'private_case_events_file':str(PRIVATE_EVENTS),
 'private_case_events_SHA256':sha256_file(PRIVATE_EVENTS),
 'not_approved_omop_or_dicom':True,'real_D_case_count':actual_people,
 'original_genuine_v15_C_not_claimed':True}
PRIVATE_PATH.write_text(json.dumps(PRIVATE_MANIFEST,indent=2,allow_nan=False),encoding='utf-8')
public={
 'project':p18['project'],'phase':'19_frozen_neuro_D_real_AB_and_synthetic_E0_E7',
 'run_utc':PRIVATE_MANIFEST['run_utc'],
 'results_status':'REAL_NEURO_D_SOURCE_TARGET_AUDIT_ON_PRIVATE_RESEARCH_CANDIDATES_PLUS_SEPARATE_SYNTHETIC_SOFTWARE_TESTS_NO_OFFICIAL_MICDM_NO_GENERIC_C',
 'privacy':'Aggregates and synthetic-only test definitions. Original private identifiers, artifact checksums and row-level events stay in Drive.',
 'inherited_phase18_A_B_checks':'PASS','original_phase17_FHIR_resource_validation':'PASS',
 'neuro_D_algorithm_contract_sha256':D_SHA256,
 'synthetic_fixture_contract_sha256':SYN_FREEZE_HASH,
 'synthetic_unit_tests':{'number_of_prespecified_cases':len(SYN_RESULTS),
    'all_passed':all(r['test_pass'] for r in SYN_RESULTS.values()),'results':SYN_RESULTS,
    'interpretation':'Engineered synthetic software validation only, not independent measured diagnostic accuracy.'},
 'actual_real_D_source_target_audit':{
    'participants_audited':actual_people,
    'arm_A':{'status':'PRIVATE_PROVENANCE_PRESERVING_CANDIDATE_NOT_STANDARD_MICDM',
       'source_supported_transformation_loss_event_counts':dict(REAL_COUNTERS['A']),
       'participants_with_each_rule':dict(REAL_FOUNDED['A']),
       'unverified_source_unit_events':UNKNOWN['A']['source_numeric_unit_unverified'],
       'unverified_DICOM_semantics_per_person_bundle':UNKNOWN['A']['source_DICOM_UID_semantics_unverified']},
    'arm_B':{'status':'INTENTIONALLY_VALUE_ONLY_STRESS_ABLATION_NOT_STANDARD_MICDM',
       'source_supported_transformation_loss_event_counts':dict(REAL_COUNTERS['B']),
       'participants_with_each_rule':dict(REAL_FOUNDED['B']),
       'unverified_source_unit_events':UNKNOWN['B']['source_numeric_unit_unverified'],
       'unverified_DICOM_semantics_per_person_bundle':UNKNOWN['B']['source_DICOM_UID_semantics_unverified']},
    'source_numeric_event_count':p18['real_input_counts']['source_numeric_Observations'],
    'source_TRACE_pair_feature_states':p18['real_input_counts']['regenerated_TRACE_pair_feature_Observations'],
    'note':'The B differences were forced by design; A parity is a software check. Neither gives independent accuracy/superiority or natural fault prevalence.'},
 'source_evidence_and_target_status':REAL_SAFETY,
 'original_frozen_generic_TFL_v15_C_actual_MRI_results':None,
 'real_four_arm_comparator_scores':None,
 'official_MICDM_load_results':None,
 'next_stage_gate':'SOURCE_MRI_STANDARD_MAPPING_AND_FROZEN_GENERIC_TFL_v15_INPUT_ADAPTER_REQUIRED_FOR_C; D_PROTOTYPE_AVAILABLE_WITH_SYNTHETIC_E0_E7_AND_REAL_PRIVATE_AB_DETECTION'}
assert public['actual_real_D_source_target_audit']['arm_A']['source_supported_transformation_loss_event_counts']=={}
assert public['real_four_arm_comparator_scores'] is None
PUBLIC_PATH.write_text(json.dumps(public,indent=2,allow_nan=False),encoding='utf-8')
print(json.dumps({'phase':public['phase'],'synthetic_cases_passed':len(SYN_RESULTS),
    'real_participants':actual_people,'A_issue_events':sum(REAL_COUNTERS['A'].values()),
    'B_intentionally_lost_image_links':REAL_COUNTERS['B']['IMAGE_LINK_LOSS_OR_MISLINK'],
    'B_intentionally_lost_pair_states':REAL_COUNTERS['B']['TRACE_PAIR_EVENT_LOSS'],
    'C':'NOT_RUN','official_MICDM':'NOT_RUN'},indent=2))


{
  "phase": "19_frozen_neuro_D_real_AB_and_synthetic_E0_E7",
  "synthetic_cases_passed": 9,
  "real_participants": 258,
  "A_issue_events": 0,
  "B_intentionally_lost_image_links": 4128,
  "B_intentionally_lost_pair_states": 1806,
  "C": "NOT_RUN",
  "official_MICDM": "NOT_RUN"
}


In [10]:
assert PUBLIC_PATH.is_file() and PRIVATE_PATH.is_file() and PRIVATE_EVENTS.is_file()
check=json.loads(PUBLIC_PATH.read_text(encoding='utf-8'))
assert check['synthetic_unit_tests']['all_passed']
assert check['actual_real_D_source_target_audit']['participants_audited']==258
assert check['real_four_arm_comparator_scores'] is None
print('Phase19 public aggregate file:',PUBLIC_PATH)
print('Send ONLY ICHI2027_Phase19_SHAREABLE.json here. Keep all private manifests and case audit events on my own Drive.')


Phase19 public aggregate file: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase19_outputs/ICHI2027_Phase19_SHAREABLE.json
Send ONLY ICHI2027_Phase19_SHAREABLE.json here. Keep all private manifests and case audit events on my own Drive.
